In [1]:
# !pip install ultralytics albumentations -q

import ultralytics
ultralytics.checks()

Ultralytics 8.4.45 🚀 Python-3.13.7 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050, 7840MiB)
Setup complete ✅ (12 CPUs, 15.5 GB RAM, 91.7/915.5 GB disk)


In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# !cp -r /content/drive/MyDrive/ikar_dataset/45_test_and_plus_lables_stratify /content/ikar_dataset_stratify_test_and_plus
# print('Датасет скопирован')

In [4]:
# Пути
DATASET_PATH = '/home/george/datasets/90_and_45_full'
RUNS_PATH    = '/home/george/datasets/90_and_45_full/yolo_runs_new_augs'
YAML_PATH    = f'{DATASET_PATH}/data.yaml'
print(YAML_PATH)

MODEL_SIZE = "s"

# Названия классов — замени если отличаются
CLASS_NAMES  = ['flange', 'obloy', 'underfill']
DEFECT_CLASSES = [1, 2]  # индексы классов дефектов (не flange)

print(f'Модель: yolov8{MODEL_SIZE}.pt')

/home/george/datasets/90_and_45_full/data.yaml
Модель: yolov8s.pt


In [5]:
import albumentations as A


def get_ablation_pipeline(experiment: str, img_size: int = 1024):

    bbox_params = A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.5
    )

    # =========================================================
    # GEOMETRY (soft OOD regularization)
    # =========================================================

    flip = [
        A.HorizontalFlip(p=0.5),
    ]

    crop = [
        A.BBoxSafeRandomCrop(
            erosion_rate=0.0,
            p=0.25
        ),
        A.Resize(img_size, img_size),
    ]

    affine = [
        A.Affine(
            scale=(0.92, 1.08),          # soft OOD (не ломаем tiny defects)
            rotate=(-5, 5),
            translate_percent=(0.02, 0.02),
            fit_output=False,
            border_mode=0,
            fill=0,
            p=0.3
        ),
    ]

    # =========================================================
    # OCCLUSION (controlled OOD)
    # =========================================================

    dropout = [
        A.ConstrainedCoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.04, 0.15),
            hole_width_range=(0.04, 0.15),
            fill="random_uniform",
            bbox_labels=[1, 2],
            p=0.15
        ),
    ]

    # =========================================================
    # LIGHTING (main robustness driver)
    # =========================================================

    lighting = [
        A.OneOf([

            A.ColorJitter(
                brightness=(0.6, 1.4),
                contrast=(0.6, 1.4),
                saturation=(0.8, 1.2),
                hue=(-0.05, 0.05),
            ),

            A.RandomBrightnessContrast(
                brightness_limit=0.4,
                contrast_limit=0.4,
            ),

            A.RandomGamma(
                gamma_limit=(65, 160),
            ),

        ], p=0.75),
    ]

    # =========================================================
    # LOCAL CONTRAST (industrial surfaces)
    # =========================================================

    clahe = [
        A.CLAHE(
            clip_limit=(1, 3),
            tile_grid_size=(8, 8),
            p=0.15
        ),
    ]

    # =========================================================
    # WORKSHOP LIGHT ARTIFACTS
    # =========================================================

    shadows = [
        A.RandomShadow(
            shadow_roi=(0, 0, 1, 1),
            num_shadows_limit=(1, 2),
            shadow_dimension=4,
            p=0.15
        ),
    ]

    # =========================================================
    # CAMERA DOMAIN SHIFT
    # =========================================================

    noise = [
        A.OneOf([

            A.GaussNoise(
                std_range=(0.01, 0.04),
                per_channel=True
            ),

            A.ISONoise(
                color_shift=(0.01, 0.05),
                intensity=(0.1, 0.4)
            ),

            A.ImageCompression(
                quality_range=(40, 95)
            ),

        ], p=0.35),
    ]

    # =========================================================
    # VERY SOFT BLUR (optional realism)
    # =========================================================

    blur = [
        A.OneOf([
            A.MotionBlur(blur_limit=3),
            A.GaussianBlur(blur_limit=(3, 3)),
        ], p=0.05),
    ]

    # =========================================================
    # PIPELINES (for ablation)
    # =========================================================

    pipelines = {

        # -------------------------------------------------
        # baseline (no augmentation)
        # -------------------------------------------------
        'baseline': [],

        # -------------------------------------------------
        # geometry only (OOD regularization)
        # -------------------------------------------------
        'geom':
            flip + crop + affine,

        # -------------------------------------------------
        # lighting robustness (MOST IMPORTANT)
        # -------------------------------------------------
        'lighting':
            flip + crop + affine + lighting,

        # -------------------------------------------------
        # lighting + camera shift
        # -------------------------------------------------
        'camera':
            flip + crop + affine + lighting + noise,

        # -------------------------------------------------
        # full realistic + soft OOD (RECOMMENDED)
        # -------------------------------------------------
        'all_soft':
            flip + crop + affine + dropout + lighting + clahe + shadows + noise + blur,

        # -------------------------------------------------
        # stronger OOD (for ablation)
        # -------------------------------------------------
        'all_strong':
            flip + crop + affine + dropout + lighting + clahe + shadows + noise,
    }

    if experiment not in pipelines:
        raise ValueError(f"Unknown experiment: {experiment}")

    return A.Compose(
        pipelines[experiment],
        bbox_params=bbox_params
    )

In [19]:
from ultralytics import YOLO
import torch
import albumentations as A
import os

# ============================================
EXPERIMENT   = 'all_strong'  # меняй только здесь
TOTAL_EPOCHS = 100
IMG_SIZE     = 1024
SEED         = 42
# ============================================

pipeline = get_ablation_pipeline(EXPERIMENT, IMG_SIZE)

print(f'Эксперимент: {EXPERIMENT}')
print(f'Pipeline: {pipeline}')

model_name = f'yolov8{MODEL_SIZE}.pt'
model = YOLO(model_name)
# model.add_callback('on_train_end', on_train_end)

# Параметры обучения
train_params = dict(
    data=YAML_PATH,
    epochs=TOTAL_EPOCHS,
    batch=8,
    imgsz=IMG_SIZE,
    optimizer='SGD',
    seed=SEED,
    device=0 if torch.cuda.is_available() else 'cpu',
    cache=False,
    workers=2,
    patience=20,

    cls=2.2,
    dfl=2.0,

    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    auto_augment=False,
    erasing=0.0,

    project=RUNS_PATH,
    name=f'ablation_{EXPERIMENT}',
    exist_ok=False,
)

# Передаём pipeline только если он есть (не baseline)
if pipeline is not None:
    # counted = CountedPipeline(pipeline)
    train_params['augmentations'] = pipeline
#print(train_params['augmentations'])

results = model.train(**train_params)
print(f'\n✅ Эксперимент [{EXPERIMENT}] завершён')

Эксперимент: all_strong
Pipeline: Compose([
  HorizontalFlip(p=0.5),
  BBoxSafeRandomCrop(p=0.25, erosion_rate=0.0),
  Resize(p=1.0, area_for_downscale=None, height=1024, interpolation=1, mask_interpolation=0, width=1024),
  Affine(p=0.3, balanced_scale=False, border_mode=0, fill=0.0, fill_mask=0.0, fit_output=False, interpolation=1, keep_ratio=False, mask_interpolation=0, rotate=(-5.0, 5.0), rotate_method='largest_box', scale={'x': (0.92, 1.08), 'y': (0.92, 1.08)}, shear={'x': (0.0, 0.0), 'y': (0.0, 0.0)}, translate_percent={'x': (0.02, 0.02), 'y': (0.02, 0.02)}, translate_px=None),
  ConstrainedCoarseDropout(p=0.15, bbox_labels=[1, 2], fill='random_uniform', fill_mask=None, hole_height_range=(0.04, 0.15), hole_width_range=(0.04, 0.15), mask_indices=None, num_holes_range=(1, 2)),
  OneOf([
    ColorJitter(p=0.5, brightness=(0.6, 1.4), contrast=(0.6, 1.4), hue=(-0.05, 0.05), saturation=(0.8, 1.2)),
    RandomBrightnessContrast(p=0.5, brightness_by_max=True, brightness_limit=(-0.4, 0.4)

/home/george/envs/ml/lib/python3.13/site-packages/albumentations/core/composition.py:465: UserWarning: transforms is single transform, but a sequence is expected! Transform will be wrapped into list.
  super().__init__(


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1665.6±392.3 MB/s, size: 128.5 KB)
val: Scanning /home/george/datasets/90_and_45_full/valid/labels.cache... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 66.0Mit/s 0.0s
optimizer: SGD(lr=0.01, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_strong/labels.jpg... 
Image sizes 1024 train, 1024 val
Using 2 dataloader workers
Logging results to /home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_strong
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      4.52G      1.351      10.81      2.108          9       1024: 100% ━━━━━━━━━━━━ 200/200 3.2it/s 1:03<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 5.9it/s 4.9s0.2s


/home/george/envs/ml/lib/python3.13/site-packages/albumentations/core/composition.py:465: UserWarning: transforms is single transform, but a sequence is expected! Transform will be wrapped into list.
  super().__init__(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     91/100      4.21G      0.807      1.857      1.413          8       1024: 100% ━━━━━━━━━━━━ 200/200 3.2it/s 1:03<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.0it/s 4.9s0.2s
                   all        456       1222      0.935      0.928       0.94      0.624

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     92/100      4.22G     0.8038      1.847      1.408         11       1024: 100% ━━━━━━━━━━━━ 200/200 3.2it/s 1:03<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.0it/s 4.9s0.2s
                   all        456       1222      0.941      0.929      0.943      0.624
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 72, best model saved as best.pt.
To upd

In [8]:
# for data in results_data:
#     for k, v in data.items():
#         print(f"{k}: {v}", sep="/n")

In [20]:

from ultralytics import YOLO

# Путь к лучшей модели
BEST_PT = '/home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_strong/weights/best.pt'
YAML_PATH = '/home/george/datasets/90_and_45_full/data.yaml'

model = YOLO(BEST_PT)

# Запускаем валидацию на test split
results = model.val(
    data=YAML_PATH,
    split='test',   # ← именно test а не val
    imgsz=1024,
    batch=16,
    device=0,
    verbose=True,
)

print('\n=== РЕЗУЛЬТАТЫ НА TEST ===')
print(f'mAP50:      {results.box.map50:.3f}')
print(f'mAP50-95:   {results.box.map:.3f}')
print(f'Precision:  {results.box.mp:.3f}')
print(f'Recall:     {results.box.mr:.3f}')

# Метрики по классам
print('\n=== ПО КЛАССАМ ===')
CLASS_NAMES = ['flange', 'obloy', 'underfill']
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:12} mAP50: {results.box.ap50[i]:.3f}  Recall: {results.box.r[i]:.3f}')

Ultralytics 8.4.45 🚀 Python-3.13.7 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050, 7840MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5300.9±2170.1 MB/s, size: 113.7 KB)
val: Scanning /home/george/datasets/90_and_45_full/test/labels.cache... 228 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 228/228 136.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.2it/s 4.7s0.3s
                   all        228        656      0.941      0.928      0.945      0.622
                flange        228        229      0.997      0.996      0.995      0.964
                 obloy        112        198      0.904      0.854      0.908      0.448
             underfill         84        229       0.92      0.934      0.932      0.453
Speed: 1.8ms preprocess, 15.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved 

In [10]:
# from ultralytics import YOLO
# import torch

# LAST_PT = '/home/george/datasets/ikar_dataset_stratify_test_and_plus/yolo_runs/ablation_noise-11/weights/last.pt'
# IMG_SIZE    = 1024

# # Создаём тот же pipeline что был при обучении
# pipeline = get_ablation_pipeline('noise', IMG_SIZE )

# model = YOLO(LAST_PT)
# model.add_callback('on_train_end', on_train_end)

# results = model.train(
#     resume=True,
#     device=0 if torch.cuda.is_available() else 'cpu',
#     augmentations=pipeline  # ← передаём явно
# )

# print('✅ Готово')

In [21]:
from ultralytics import YOLO

model = YOLO('/home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_soft/weights/best.pt',
    task='detect')
model.export(
    format='engine',
    device=0,
    imgsz=1024,
    half=True,      # FP16 — в 2 раза быстрее
    batch=1,        # для реального времени
)
# Создаст файл best.engine

Ultralytics 8.4.45 🚀 Python-3.13.7 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050, 7840MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_soft/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 7, 21504) (21.6 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 0.8s, saved as '/home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_soft/weights/best.onnx' (42.9 MB)

TensorRT: starting export with TensorRT 10.16.1.11...
[05/08/2026-01:02:25] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 3721, GPU 2548 (MiB)
[05/08/2026-01:02:26] [TRT] [I] ----------------------------------------------------------------
[05/08/2026-01:02:26] [TRT] [I] Input filename:   /home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_soft/weight

PosixPath('/home/george/datasets/90_and_45_full/yolo_runs_new_augs/ablation_all_soft/weights/best.engine')

In [11]:
from ultralytics import YOLO
import time

# Загружаем TensorRT модель
model_trt = YOLO('/home/george/datasets/90_and_45_full/yolo_runs/ablation_noise-4/weights/best.engine')

# Тестовое изображение
test_image = '/home/george/datasets/ikar_dataset_stratify_test_and_plus/test/images/IMG_0051_JPG_JPG_JPG.rf.12d942ad38ae71a651781a5b9ddc190d.jpg'

# Прогрев модели (первый запуск всегда медленнее)
model_trt(test_image, imgsz=1024, device=0, verbose=False)
model_trt(test_image, imgsz=1024, device=0, verbose=False)

# Замер скорости на 20 кадрах
times = []
for i in range(20):
    start = time.time()
    results = model_trt(test_image, imgsz=1024, device=0, verbose=False)
    elapsed = time.time() - start
    times.append(elapsed * 1000)

avg_ms  = sum(times) / len(times)
avg_fps = 1000 / avg_ms

print(f'Среднее время: {avg_ms:.1f} ms')
print(f'Средний FPS:   {avg_fps:.1f}')
print(f'Мин: {min(times):.1f} ms  Макс: {max(times):.1f} ms')

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading /home/george/datasets/90_and_45_full/yolo_runs/ablation_noise-4/weights/best.engine for TensorRT inference...
[05/06/2026-21:11:18] [TRT] [I] Loaded engine size: 24 MiB
[05/06/2026-21:11:18] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +43, now: CPU 0, GPU 64 (MiB)
Среднее время: 15.3 ms
Средний FPS:   65.3
Мин: 15.2 ms  Макс: 15.5 ms


In [2]:
from ultralytics import YOLO
import time

# Загружаем TensorRT модель
model_trt = YOLO('/home/george/datasets/ikar_dataset_stratify_test_and_plus/yolo_runs/ablation_noise-13_best/weights/best.engine')

# Тестовое изображение
test_image = '/home/george/datasets/ikar_dataset_stratify_test_and_plus/test/images/IMG_0051_JPG_JPG_JPG.rf.12d942ad38ae71a651781a5b9ddc190d.jpg'


# Прогрев модели (первый запуск всегда медленнее)
model_trt(test_image, imgsz=1024, device=0, verbose=False)
model_trt(test_image, imgsz=1024, device=0, verbose=False)

# Замер скорости на 20 кадрах
times = []
for i in range(20):
    start = time.time()
    results = model_trt(test_image, imgsz=1024, device=0, verbose=False)
    elapsed = time.time() - start
    times.append(elapsed * 1000)

avg_ms  = sum(times) / len(times)
avg_fps = 1000 / avg_ms

print(f'Среднее время: {avg_ms:.1f} ms')
print(f'Средний FPS:   {avg_fps:.1f}')
print(f'Мин: {min(times):.1f} ms  Макс: {max(times):.1f} ms')

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading /home/george/datasets/ikar_dataset_stratify_test_and_plus/yolo_runs/ablation_noise-13_best/weights/best.engine for TensorRT inference...
[05/04/2026-15:07:28] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
[05/04/2026-15:07:28] [TRT] [I] Loaded engine size: 24 MiB
[05/04/2026-15:07:28] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +1, GPU +43, now: CPU 1, GPU 128 (MiB)
Среднее время: 15.2 ms
Средний FPS:   65.6
Мин: 15.0 ms  Макс: 17.3 ms
